https://github.com/ijstokes/tutorial-local-llm

---

# Local LLM

---

In [93]:
# load in libraries
import json
from ollama import Client, chat
from pathlib import Path
import pandas as pd

In [94]:
import subprocess
import time

# Start the Ollama server if it is not already running
ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give the server a moment to start
time.sleep(3)

# Check whether the server is running
print("Ollama process:", ollama_server.poll())

Ollama process: None


In [95]:
from ollama import list as list_models

# Check which models are available
models = list_models()

print(models)

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 57, 137259, tzinfo=TzInfo(39600)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')), Model(model='codellama:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 51, 300149, tzinfo=TzInfo(39600)), digest='8fdf8f752f6e80de33e82f381aba784c025982752cd1ae9377add66449d2225f', size=3825910662, details=ModelDetails(parent_model='', format='gguf', family='llama', families=None, parameter_size='7B', quantization_level='Q4_0')), Model(model='mistral:latest', modified_at=datetime.datetime(2026, 10, 27, 7, 43, 46, 29050, tzinfo=TzInfo(39600)), digest='6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', size=4372824384, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], 

In [ ]:
CONDITION = "aig"

# Set the path to the processed passages
DATA_DIR = Path("..") / "pmc" / CONDITION / "processed"

Articles: 5
Passages: 10


,pmcid,text,passage_ids,n_passages,max_score
0,PMC11010983,The prevalence of AAG in the general populatio...,"[5, 17]",2,69.326569
1,PMC11863545,The objective of this retrospective cohort stu...,[6],1,69.230820
2,PMC3520153,The cut-off values adopted were those suggeste...,[8],1,69.347687
3,PMC6502205,"In practical terms, it is advisable to screen ...",[16],1,69.181526
4,PMC8621657,CAG can be asymptomatic or can be characterize...,"[9, 20, 26, 27, 28]",5,70.686111


### Full-text Articles

In [ ]:
# Load all full-text passages
all_passages = pd.read_parquet(
    DATA_DIR / "passages.parquet"
)

# Group all passages by article
fulltext_articles = (
    all_passages.sort_values(["pmcid", "passage_id"])
    .groupby("pmcid", as_index=False)
    .agg(
        text=("text", lambda x: "\n\n".join(x.dropna())),
        passage_ids=("passage_id", list),
        n_passages=("passage_id", "size")
    )
)

# Check the number of articles and passages
print("Articles:", len(fulltext_articles))
print("Passages:", fulltext_articles["n_passages"].sum())

display(fulltext_articles.head())

### MedCPT

In [ ]:
# Load the diagnostic passages grouped by article
diagnostic_articles = pd.read_parquet(
    DATA_DIR / "diagnostic_articles.parquet"
)

# Check the number of articles and passages
print("Articles:", len(diagnostic_articles))
print("Passages:", diagnostic_articles["n_passages"].sum())

display(diagnostic_articles)

### BERTopic

## Header

In [97]:
# Connect to the local Ollama server
client = Client(host="http://localhost:11434")

# select the local llm to use
MODEL = "llama3.2"

In [98]:
# Create instructions for extracting diagnostic criteria
PROMPT = """
Extract diagnostic criteria for autoimmune gastritis (AIG)
from the provided medical passages.

Only extract information explicitly stated in the source text.
Do not invent diagnostic criteria, tests or thresholds.

Do not extract symptoms, complications, prevalence statistics
or screening recommendations unless explicitly described
as diagnostic criteria.

Return a JSON object containing a list named "criteria".

For each diagnostic criterion, extract:
- condition: the disease being diagnosed
- criterion: description of the diagnostic criterion
- category: serological, histological, endoscopic, clinical,
  microbiological, imaging or other
- criteria_type: general diagnostic criterion, study-specific
  diagnostic criterion or unspecified
- diagnostic_test: test used to identify the criterion
- marker: biomarker or characteristic being assessed
- threshold: numerical cut-off including comparison operator
- unit: measurement unit
- diagnostic_role: confirmatory, supportive, exclusionary
  or unspecified
- combination: other diagnostic criteria required in combination
- evidence: exact supporting quotation from the source text

Follow these rules:
- Preserve numerical thresholds and their measurement units.
- Preserve AND/OR relationships between diagnostic criteria.
- Do not assume that individual findings confirm a diagnosis.
- Ensure the evidence directly supports the extracted criterion.
- Copy evidence verbatim without paraphrasing.
- Use null for missing information, not the string "None".
- Only provide units for numerical measurements.

Return valid JSON only, without Markdown or commentary.

If no diagnostic criteria are found, return:
{{"criteria": []}}

Passages:
{passage}
"""

In [99]:
prompt = PROMPT

In [100]:
# Create a function to send prompts to the local LLM
def run_llm(prompt, model=MODEL):

    # Send the prompt to the local LLM
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 16384
        }
    )

    # Return the generated response
    return response.message.content, response.done_reason

In [ ]:
# Create a function to extract diagnostic criteria from articles
def extract_articles(articles_df, method, model=MODEL):

    # Create a list to store the extraction results
    results = []

    # Go through each article
    for _, row in articles_df.iterrows():

        PMCID = row["pmcid"]
        passage = row["text"]

        # Insert the article text into the diagnostic prompt
        prompt = PROMPT.format(passage=passage)

        # Extract diagnostic criteria
        result = run_llm(prompt, model=model)

        # Store the extraction results
        results.append({
            "pmcid": PMCID,
            "model": model,
            "retrieval_method": method,
            "n_passages": int(row["n_passages"]),
            "passage_ids": [
                int(pid) for pid in row["passage_ids"]
            ],
            "response": result
        })

        print("Processed:", PMCID)

    return results

Processed: PMC11010983
Processed: PMC11863545
Processed: PMC3520153
Processed: PMC6502205
Processed: PMC8621657


In [ ]:
medcpt_results = extract_articles(
    diagnostic_articles,
    method="medcpt"
)

In [ ]:
fulltext_results = extract_articles(
    fulltext_articles,
    method="full_text"
)

In [ ]:
# Create a function to save the extraction results as JSON
def save_results(results, method, output_dir=DATA_DIR):

    # Convert results into a JSON file
    with open(
        output_dir / f"{method}_llm_results.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

In [ ]:
save_results(medcpt_results, "medcpt")
save_results(fulltext_results, "full_text")

In [ ]:
# Create a function to display the LLM extraction results
def display_results(results):

    # Convert the results into a DataFrame
    results_df = pd.DataFrame(results)

    display(results_df)

    # Display the extracted diagnostic criteria for each article
    for result in results:

        print("Article:", result["pmcid"])

        # Get the generated text from the LLM response
        response = result["response"]

        # Handle responses containing text and completion reason
        if isinstance(response, (list, tuple)):
            response = response[0]

        # Display the response
        print(response)
        print("-" * 80)

    return results_df

,pmcid,model,retrieval_method,n_passages,passage_ids,response
0,PMC11010983,llama3.2,medcpt,2,"[5, 17]","({""criteria"": [\n {\n ""condition"": ""autoimm..."
1,PMC11863545,llama3.2,medcpt,1,[6],"({""criteria"": [\n {\n ""condition"": ""Autoimm..."
2,PMC3520153,llama3.2,medcpt,1,[8],"({""criteria"": [\n {\n ""condition"": ""Autoimm..."
3,PMC6502205,llama3.2,medcpt,1,[16],"({""criteria"": [\n {\n ""condition"": ""autoimm..."
4,PMC8621657,llama3.2,medcpt,5,"[9, 20, 26, 27, 28]","({""criteria"": [\n {\n ""condition"": ""Autoimm..."


In [ ]:
medcpt_df = display_results(medcpt_results)

Article: PMC11010983
('{"criteria": [\n  {\n   "condition": "autoimmune gastritis",\n   "criterion": "histological examination of gastric biopsy specimens",\n   "category": "histological",\n   "criteria_type": "study-specific diagnostic criterion",\n   "diagnostic_test": "histological examination",\n   "marker": "absence of a honeycomb subepithelial capillary network and a regular-shape venous collection",\n   "threshold": null,\n   "unit": null,\n   "diagnostic_role": "confirmatory",\n   "combination": null,\n   "evidence": "In normal conditions, micro-vessels of the gastric mucosa of the body show a honeycomb subepithelial capillary network and a regular-shape venous collection, which are absent in patients with atrophic gastritis [37]."\n  },\n  {\n   "condition": "autoimmune gastritis",\n   "criterion": "low levels of Vitamin B12",\n   "category": "serological",\n   "criteria_type": "general diagnostic criterion",\n   "diagnostic_test": "serological test",\n   "marker": "low levels

In [ ]:
fulltext_df = display_results(fulltext_results)

Article: PMC11010983
{"criteria": [
  {
   "condition": "autoimmune gastritis",
   "criterion": "histological examination of gastric biopsy specimens",
   "category": "histological",
   "criteria_type": "study-specific diagnostic criterion",
   "diagnostic_test": "histological examination",
   "marker": "absence of a honeycomb subepithelial capillary network and a regular-shape venous collection",
   "threshold": null,
   "unit": null,
   "diagnostic_role": "confirmatory",
   "combination": null,
   "evidence": "In normal conditions, micro-vessels of the gastric mucosa of the body show a honeycomb subepithelial capillary network and a regular-shape venous collection, which are absent in patients with atrophic gastritis [37]."
  },
  {
   "condition": "autoimmune gastritis",
   "criterion": "low levels of Vitamin B12",
   "category": "serological",
   "criteria_type": "general diagnostic criterion",
   "diagnostic_test": "serological test",
   "marker": "low levels of Vitamin B12",
   "

In [ ]:
#bertopic_df = display_results(bertopic_results)

In [ ]:
# Stop the Ollama server
ollama_server.terminate()
ollama_server.wait()


0